In [2]:
import os
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from typing import List
from functools import partial

from src.components.llm import OpenAILLM
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pydantic import BaseModel, Field
import textwrap
import json

In [3]:
judge_client = OpenAILLM(
                    model="gpt-4.1-mini", 
                    base_url="https://aihubmix.com/v1", 
                    api_key="sk-XWaGp10Cjy2pZfttA8E538967f7f4dA7A463F584C17b63Bf", 
                )

In [46]:
from transformers import AutoModelForSequenceClassification

# Step 1: Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    'vectara/hallucination_evaluation_model', trust_remote_code=True)



You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


In [15]:
import ast

def parse_llm_list_output(llm_output_string: str) -> list:
    """
    清洗并解析被Markdown代码块和引号包裹的LLM输出字符串，
    将其安全地转换为Python列表。
    """
    
    # 1. 移除外层的单引号和前后空格
    cleaned_string = llm_output_string.strip().strip("'")
    
    # 2. 移除Markdown代码块的起始和结束标记
    # 标记包括：```python\n 和 \n```
    if cleaned_string.startswith('```python'):
        # 移除起始标记 '```python\n' (使用 replace 确保只替换第一个)
        cleaned_string = cleaned_string.replace('```python\n', '', 1).strip()
    
    if cleaned_string.endswith('```'):
        # 移除结束标记 '\n```'
        cleaned_string = cleaned_string.rstrip('```').strip()
    
    # 3. 使用 ast.literal_eval 安全地将字符串解析为Python对象（列表）
    try:
        # ast.literal_eval 比 eval() 安全，因为它只评估字面量（如字符串、数字、列表、字典）
        data_list = ast.literal_eval(cleaned_string)
        
        # 确保解析结果确实是一个列表
        if isinstance(data_list, list):
            return data_list
        else:
            print("解析失败：字符串内容不是一个有效的列表结构。")
            return []
            
    except (ValueError, SyntaxError) as e:
        print("解析失败：字符串包含无效的Python语法或格式错误。")
        print(f"错误信息: {e}")
        # 打印部分字符串帮助调试
        print(f"尝试解析的字符串片段: {cleaned_string[:500]}...")
        return []

judge_prompt = textwrap.dedent("""
                            [任务]
                            你是一个将文本分解成原子主张的专家。请严格按照用户要求，仅输出列表格式。你的任务是将下面的回答（Answer）分解成一系列独立、原子化的事实性陈述或主张（Claims）。每个 Claim 都必须是可独立验证的。

                            [输入]
                            Answer: {answer}

                            [输出要求]
                            请以一个 Python 列表，输出所有提取出的 Claims。

                            例如：
                            Input: "亚历山大·弗莱明于1928年发现了青霉素，这标志着抗生素时代的开始。"
                            Output: ["亚历山大·弗莱明发现了青霉素。", "青霉素发现于1928年。", "青霉素的发现标志着抗生素时代的开始。"]
                        """)

In [16]:
def jsonl_results_loader(save_path,num_records=500):
    """加载 JSONL 格式的结果文件"""
    results = []
    with open(save_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            results.append(record)
    return results[0:num_records]

In [17]:
out = jsonl_results_loader("/mnt/data1/workspace/zms/LeakDojo/results/fiqa_4ragas/qwen3-32b/R__bge-large-en-v1_5_k10-RR__bge-reranker-large_n5-EX__bge-large-en-v1_5/WBTQ_RW-0_RR-1_EX-0_IF-0_OF-0_none_0_4ragas.jsonl")
id= 0

In [28]:
out[id]

{'id': '10385',
 'adversarial_template': '{text}',
 'query_with_template': 'When should I start saving/investing for my retirement?',
 'cleaned_query': 'When should I start saving/investing for my retirement?',
 'rewritten_queries': [None],
 'contexts': ['"You will hear a lot about diversifying your portfolio, which typically means having a good mix of investment types, areas of investments, etc. I\'d like to suggest that you should also diversify your sources.  Sad to say but the defined benefit pension is not a rock solid, sure fire source of security in your retirement planning. Companies go bankrupt, government agencies are reorganized, and those hitherto-untouchable assets are destroyed overnight. So, treat your new investment strategy as if you were starting over, and invest accordingly, for example, aggressively for a few years, then progressively safer as you get older. There are other strategies too, depending on factors like your taste for risk: you might prefer to be conserv

In [18]:
prompt = judge_prompt.format(answer=out[id]["answer"])
prompt

'\n[任务]\n你是一个将文本分解成原子主张的专家。请严格按照用户要求，仅输出列表格式。你的任务是将下面的回答（Answer）分解成一系列独立、原子化的事实性陈述或主张（Claims）。每个 Claim 都必须是可独立验证的。\n\n[输入]\nAnswer: **When to Start Saving/Investing for Retirement**  \nStart **as soon as you’ve built a basic emergency fund** (6 months of income) *and* secured your education or career growth. Here’s a simplified breakdown:\n\n---\n\n### **1. Prioritize Education First**  \n- **Why?** A college degree or higher education can earn you **$400,000–$1 million more over your lifetime** than a high school diploma. This is a guaranteed "investment" that pays off long-term.  \n- **Action:** Focus on completing your education before aggressively saving for retirement. Your future higher income will make retirement savings easier later.\n\n---\n\n### **2. Build an Emergency Fund (6 Months of Income)**  \n- **Why?** Without this, unexpected expenses (e.g., car repairs, medical bills) could force you to dip into retirement savings or take on debt.  \n- **How much?** For you, 6 month

In [22]:
judge_out = []
response, _ = judge_client.infer(prompt)

In [23]:
response

'[\n    "在建立基本的紧急基金（6个月的收入）并确保教育或职业发展后，应尽快开始为退休储蓄或投资。",\n    "大学学位或更高教育可以使你在一生中比仅有高中毕业证多赚40万至100万美元。",\n    "完成教育是一个保证的长期投资回报。",\n    "应优先完成教育，然后再积极为退休储蓄。",\n    "未来更高的收入将使退休储蓄变得更容易。",\n    "没有紧急基金时，意外开支（如汽车维修、医疗账单）可能迫使你动用退休储蓄或负债。",\n    "建议的紧急基金金额是6个月的收入。",\n    "6个月的收入等于6000美元，基于每周收入200美元计算。",\n    "当前储蓄为1000美元。",\n    "每周储蓄150美元，约8个月后可达到6000美元的紧急基金目标。",\n    "在完成教育并建立紧急基金后，应该开始退休储蓄。",\n    "退休储蓄可通过Roth IRA进行，每年可贡献最高5500美元。",\n    "Roth IRA提供免税增长和灵活取款的优势。",\n    "应投资指数基金，如标普500指数基金，这些是多样化且低成本的选择。",\n    "除非是专家，否则应避免投资个股。",\n    "随着收入增长，应调整退休储蓄比例。",\n    "举例来说，若每周收入200美元（年收入10400美元），可每周储蓄150美元。",\n    "在这150美元中，100美元用于退休储蓄（Roth IRA或应税账户），50美元用于偿还学生贷款或扩大紧急基金。",\n    "应尽早开始储蓄以利用复利效应，但不能以牺牲教育或财务稳定为代价。",\n    "Roth IRA适合年轻储户，因其税收优惠和灵活性。",\n    "指数基金比挑选个股更安全且成本更低。",\n    "示例时间表：前8个月储蓄6000美元作为紧急基金。",\n    "示例时间表：完成教育后，开始每年向Roth IRA投资5500美元，主要投资指数基金。",\n    "示例时间表：长期来看，随着收入增长，增加退休储蓄额度。",\n    "优先考虑教育和紧急储蓄，有助于实现更明智、更可持续的退休规划。"\n]'

In [34]:
aaa = parse_llm_list_output(response)
claims = aaa

In [35]:
claims

['在建立基本的紧急基金（6个月的收入）并确保教育或职业发展后，应尽快开始为退休储蓄或投资。',
 '大学学位或更高教育可以使你在一生中比仅有高中毕业证多赚40万至100万美元。',
 '完成教育是一个保证的长期投资回报。',
 '应优先完成教育，然后再积极为退休储蓄。',
 '未来更高的收入将使退休储蓄变得更容易。',
 '没有紧急基金时，意外开支（如汽车维修、医疗账单）可能迫使你动用退休储蓄或负债。',
 '建议的紧急基金金额是6个月的收入。',
 '6个月的收入等于6000美元，基于每周收入200美元计算。',
 '当前储蓄为1000美元。',
 '每周储蓄150美元，约8个月后可达到6000美元的紧急基金目标。',
 '在完成教育并建立紧急基金后，应该开始退休储蓄。',
 '退休储蓄可通过Roth IRA进行，每年可贡献最高5500美元。',
 'Roth IRA提供免税增长和灵活取款的优势。',
 '应投资指数基金，如标普500指数基金，这些是多样化且低成本的选择。',
 '除非是专家，否则应避免投资个股。',
 '随着收入增长，应调整退休储蓄比例。',
 '举例来说，若每周收入200美元（年收入10400美元），可每周储蓄150美元。',
 '在这150美元中，100美元用于退休储蓄（Roth IRA或应税账户），50美元用于偿还学生贷款或扩大紧急基金。',
 '应尽早开始储蓄以利用复利效应，但不能以牺牲教育或财务稳定为代价。',
 'Roth IRA适合年轻储户，因其税收优惠和灵活性。',
 '指数基金比挑选个股更安全且成本更低。',
 '示例时间表：前8个月储蓄6000美元作为紧急基金。',
 '示例时间表：完成教育后，开始每年向Roth IRA投资5500美元，主要投资指数基金。',
 '示例时间表：长期来看，随着收入增长，增加退休储蓄额度。',
 '优先考虑教育和紧急储蓄，有助于实现更明智、更可持续的退休规划。']

In [47]:
# 假设 pipe 对象已在外部成功加载
# 假设 out[id]["contexts"] 是一个包含独立 Context 字符串的列表
# 假设 claims 是一个包含 Answer 中所有事实性声明的列表

entailment_count = 0
total_claims = len(claims)

# 确保 contexts_list 已经正确处理，使其成为一个可遍历的列表
context_list = out[id]["contexts"]
if not isinstance(context_list, list):
    # 如果它是一个长字符串，您需要在这里定义如何分割它，例如：
    # context_list = raw_contexts.split('\n\n')
    # 鉴于您之前的数据，这里假设它是一个列表
    pass

# --- 外层循环：遍历 Answer 中的每一个 Claim ---
for claim in claims:
    
    # 记录该 claim 是否在所有 contexts 中找到了 ENTAILMENT
    is_entailed_by_any_context = False
    
    # --- 内层循环：遍历检索到的每一个 Context ---
    for context in context_list:
        
        # 排除空字符串的 Context
        if not context.strip():
            continue
            
        try:
            # 使用 pipeline 进行推理
            # HHEM 模型通常使用 Context 作为 Premise (前提)，Claim 作为 Hypothesis (假设)
            # pipeline 接口会自动处理这两个字符串的拼接和编码
            print("Using pipeline for verification...")
            # print("Context:", context)
            # print("Claim:", claim)

            # pipeline 调用：将前提和假设作为列表中的两个元素传入
            # results = pipe(
            #     [(context, claim)]
            # )
            
            # Step 2: Use the model to predict
            results = model.predict( [(context, claim)]) # note the predict() method. Do not do model(pairs). 
            # tensor([0.0111, 0.6474, 0.1290, 0.8969, 0.1846, 0.0050, 0.0543])
            
            # 提取预测标签
            # results 示例: [{'label': 'ENTAILMENT', 'score': 0.99}]
            predicted_label = results[0]['label']
            
            # 4. 最大值原则：只要发现一个 ENTAILMENT，就停止对该 Claim 的 Context 遍历
            if predicted_label == "ENTAILMENT":
                is_entailed_by_any_context = True
                print("--- ENTAILMENT FOUND ---")
                break # 找到了支持，跳出内层循环，进入下一个 Claim
            else:
                 print(f"Prediction: {predicted_label}")
                
        except Exception as e:
            # 捕获推理过程中的任何错误，并继续下一个 Context
            print(f"Pipeline 推理发生错误: {e}")
            continue

    # --- 外部判断：更新 entailment_count ---
    if is_entailed_by_any_context:
        entailment_count += 1
    
# --- 计算最终分数 ---
faithfulness_score = entailment_count / total_claims if total_claims else 0

# --- 记录结果 ---
judge_out.append({
    "total_claims": total_claims,
    "entailed_claims": entailment_count,
    "faithfulness_score": faithfulness_score
})

print(f"\nFinal Faithfulness Score: {faithfulness_score:.4f}")

Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...


Token indices sequence length is longer than the specified maximum sequence length for this model (1115 > 512). Running this sequence through the model will result in indexing errors


Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of dimension 0
Using pipeline for verification...
Pipeline 推理发生错误: too many indices for tensor of di

KeyboardInterrupt: 

In [37]:
judge_out

[{'total_claims': 25, 'entailed_claims': 0, 'faithfulness_score': 0.0},
 {'total_claims': 25, 'entailed_claims': 0, 'faithfulness_score': 0.0},
 {'total_claims': 25, 'entailed_claims': 0, 'faithfulness_score': 0.0},
 {'total_claims': 25, 'entailed_claims': 0, 'faithfulness_score': 0.0}]

In [31]:
faithfulness_score

0.0

In [5]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import FaithfulnesswithHHEM
from langchain.chat_models import ChatOpenAI
from ragas.llms import LangchainLLMWrapper


llm = ChatOpenAI(
        base_url="https://aihubmix.com/v1",
        api_key="sk-XWaGp10Cjy2pZfttA8E538967f7f4dA7A463F584C17b63Bf",
        model="gpt-4.1-mini", timeout=300.0
)

evaluator_llm = LangchainLLMWrapper(llm)

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = FaithfulnesswithHHEM(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

/tmp/ipykernel_265338/2162815222.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
/tmp/ipykernel_265338/2162815222.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


0.0